Importacion de librerias

In [6]:
import pandas as pd
import numpy as np
import io
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import LabelEncoder, MinMaxScaler
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import classification_report

print("Listo, librerias cargadas")

Listo, librerias cargadas


Generacion del DATASET

In [7]:
np.random.seed(42)
n = 200

productos     = ['Chipa', 'Sopa paraguaya', 'Mbeju', 'Chipa guasu', 'Pastel mandio']
departamentos = ['Central', 'Asuncion', 'Cordillera', 'Paraguari', 'Itapua']
vendedores    = ['Don Beto', 'Dona Rosa', 'Carlitos', 'La Pety', 'El Chuky']

data = {
    'producto'      : np.random.choice(productos, n),
    'departamento'  : np.random.choice(departamentos, n),
    'vendedor'      : np.random.choice(vendedores, n),
    'precio_gs'     : np.random.randint(1000, 50000, n).astype(float),
    'cantidad'      : np.random.randint(1, 100, n).astype(float),
    'temperatura_c' : np.random.uniform(18, 42, n),
    'venta_alta'    : None
}

df_raw = pd.DataFrame(data)

ingreso = df_raw['precio_gs'] * df_raw['cantidad']
df_raw['venta_alta'] = (ingreso > ingreso.median()).astype(int)

idx_nan_precio = np.random.choice(n, 15, replace=False)
idx_nan_cant   = np.random.choice(n, 10, replace=False)
df_raw.loc[idx_nan_precio, 'precio_gs'] = np.nan
df_raw.loc[idx_nan_cant,   'cantidad']  = np.nan

idx_case = np.random.choice(n, 20, replace=False)
df_raw.loc[idx_case, 'departamento'] = df_raw.loc[idx_case, 'departamento'].str.upper()

df_raw.loc[[5, 77], 'temperatura_c'] = [999, -50]

csv_buffer = io.StringIO()
df_raw.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)

print(f"Dataset generado: {df_raw.shape[0]} filas y {df_raw.shape[1]} columnas")
print(f"Valores faltantes:\n{df_raw.isnull().sum()}")
df_raw.head(8)

Dataset generado: 200 filas y 7 columnas
Valores faltantes:
producto          0
departamento      0
vendedor          0
precio_gs        15
cantidad         10
temperatura_c     0
venta_alta        0
dtype: int64


,producto,departamento,vendedor,precio_gs,cantidad,temperatura_c,venta_alta
0,Chipa guasu,Asuncion,Carlitos,12003.0,47.0,32.978437,0
1,Pastel mandio,Cordillera,El Chuky,22732.0,86.0,30.954745,1
2,Mbeju,Central,Don Beto,26826.0,56.0,28.529872,1
3,Pastel mandio,Central,El Chuky,31354.0,94.0,31.859671,1
4,Pastel mandio,PARAGUARI,La Pety,14843.0,63.0,26.528699,1
5,Sopa paraguaya,Cordillera,El Chuky,NaN,48.0,999.000000,1
6,Mbeju,Itapua,Don Beto,49529.0,61.0,30.764580,1
7,Mbeju,Cordillera,La Pety,7190.0,NaN,19.598859,0


Primer Agente

In [8]:
# Se encarga de limpiar el dataset antes de usarlo
class AgenteNormalizador:

    def __init__(self):
        self.encoders = {}
        self.scaler   = MinMaxScaler()
        self.acciones = []

    def limpiar_texto(self, df):
        for col in df.select_dtypes(include='object').columns:
            df[col] = df[col].str.strip().str.lower()
        self.acciones.append("Texto puesto en minusculas y sin espacios extra")
        return df

    def sacar_outliers(self, df):
        raros = df['temperatura_c'].apply(lambda x: x < 0 or x > 60).sum()
        df['temperatura_c'] = df['temperatura_c'].apply(
            lambda x: np.nan if (x < 0 or x > 60) else x
        )
        self.acciones.append(f"Temperaturas imposibles eliminadas: {raros}")
        return df

    def rellenar_vacios(self, df):
        for col in df.select_dtypes(include=[np.number]).columns:
            vacios = df[col].isnull().sum()
            if vacios > 0:
                mediana = df[col].median()
                df[col] = df[col].fillna(mediana)
                self.acciones.append(f"Columna '{col}': {vacios} vacios rellenados con {mediana:.1f}")
        return df

    def convertir_texto_a_numero(self, df, columnas):
        for col in columnas:
            le = LabelEncoder()
            df[col] = le.fit_transform(df[col].astype(str))
            self.encoders[col] = le
        self.acciones.append(f"Columnas convertidas a numero: {columnas}")
        return df

    def escalar(self, df, columnas):
        df[columnas] = self.scaler.fit_transform(df[columnas])
        self.acciones.append(f"Numeros escalados entre 0 y 1: {columnas}")
        return df

    def ejecutar(self, csv_buffer):
        print("=" * 50)
        print("  AGENTE 1 - NORMALIZADOR")
        print("=" * 50)

        df = pd.read_csv(csv_buffer)
        print(f"Dataset recibido: {df.shape}")

        cols_texto  = ['producto', 'departamento', 'vendedor']
        cols_num    = ['precio_gs', 'cantidad', 'temperatura_c']
        col_destino = 'venta_alta'

        df = self.limpiar_texto(df)
        df = self.sacar_outliers(df)
        df = self.rellenar_vacios(df)
        df = self.convertir_texto_a_numero(df, cols_texto)
        df = self.escalar(df, cols_num)

        print("\nCosas que hizo el agente:")
        for a in self.acciones:
            print(f"  - {a}")

        print(f"\nDataset limpio listo: {df.shape}")
        return df, col_destino

agente1 = AgenteNormalizador()
csv_buffer.seek(0)
df_limpio, target = agente1.ejecutar(csv_buffer)

print("\nPrimeras filas del dataset limpio:")
df_limpio.head()

  AGENTE 1 - NORMALIZADOR
Dataset recibido: (200, 7)

Cosas que hizo el agente:
  - Texto puesto en minusculas y sin espacios extra
  - Temperaturas imposibles eliminadas: 2
  - Columna 'precio_gs': 15 vacios rellenados con 24793.0
  - Columna 'cantidad': 10 vacios rellenados con 47.5
  - Columna 'temperatura_c': 2 vacios rellenados con 30.8
  - Columnas convertidas a numero: ['producto', 'departamento', 'vendedor']
  - Numeros escalados entre 0 y 1: ['precio_gs', 'cantidad', 'temperatura_c']

Dataset limpio listo: (200, 7)

Primeras filas del dataset limpio:


,producto,departamento,vendedor,precio_gs,cantidad,temperatura_c,venta_alta
0,1,0,0,0.226587,0.469388,0.620908,0
1,3,2,3,0.447712,0.867347,0.535346,1
2,2,1,1,0.532090,0.561224,0.432822,1
3,3,1,3,0.625412,0.948980,0.573606,1
4,3,4,4,0.285120,0.632653,0.348212,1


Segundo Agente

In [9]:
# Prueba varios modelos y elige el que mejor funciona
class AgenteEntrenador:

    def __init__(self):
        self.modelos = {
            'Arbol de Decision' : DecisionTreeClassifier(max_depth=4, random_state=42),
            'KNN (vecinos)'     : KNeighborsClassifier(n_neighbors=5),
            'Naive Bayes'       : GaussianNB()
        }
        self.resultados   = {}
        self.mejor_modelo = None
        self.mejor_nombre = ''
        self.reporte      = ''

    def ejecutar(self, df, col_destino):
        print("=" * 50)
        print("  AGENTE 2 - ENTRENADOR")
        print("=" * 50)

        X = df.drop(columns=[col_destino]).values
        y = df[col_destino].values

        print(f"Columnas usadas como entrada: {X.shape[1]}")
        print(f"Total de ejemplos: {X.shape[0]}")
        print(f"Ventas bajas: {(y==0).sum()} | Ventas altas: {(y==1).sum()}")
        print()

        for nombre, modelo in self.modelos.items():
            puntuaciones = cross_val_score(modelo, X, y, cv=5, scoring='accuracy')
            self.resultados[nombre] = puntuaciones.mean()
            print(f"  {nombre:<22}  precision promedio: {puntuaciones.mean():.4f}")

        self.mejor_nombre = max(self.resultados, key=self.resultados.get)
        self.mejor_modelo = self.modelos[self.mejor_nombre]
        self.mejor_modelo.fit(X, y)

        y_pred = self.mejor_modelo.predict(X)
        self.reporte = classification_report(y, y_pred, target_names=['Baja', 'Alta'])

        print(f"\nModelo elegido: {self.mejor_nombre}")
        print(f"Precision: {self.resultados[self.mejor_nombre]:.4f}")
        print("Entrenamiento terminado.")

        metricas = {
            'nombre_modelo'  : self.mejor_nombre,
            'precision'      : self.resultados[self.mejor_nombre],
            'todos'          : self.resultados,
            'reporte'        : self.reporte,
            'n_ejemplos'     : X.shape[0],
            'n_columnas'     : X.shape[1]
        }
        return metricas, self.mejor_modelo

agente2 = AgenteEntrenador()
metricas, modelo_final = agente2.ejecutar(df_limpio, target)

  AGENTE 2 - ENTRENADOR
Columnas usadas como entrada: 6
Total de ejemplos: 200
Ventas bajas: 100 | Ventas altas: 100

  Arbol de Decision       precision promedio: 0.9150
  KNN (vecinos)           precision promedio: 0.6500
  Naive Bayes             precision promedio: 0.9000

Modelo elegido: Arbol de Decision
Precision: 0.9150
Entrenamiento terminado.
